# Core concepts

This notebook explains what a Phylo2Vec vector encodes, and how it is converted to and from other tree representations:
* Phylogenetic trees and the size of tree space
* Why encode trees as vectors
* The Phylo2Vec vector, and ordered and unordered vectors
* Other tree formats used by Phylo2Vec
* Converting a vector to a tree, and a Newick string to a vector
* Branch lengths: the matrix format
* The complexity of these conversions

It assumes you have gone through [Getting started](getting-started.ipynb). The text and figures are adapted from the [PhD thesis](https://github.com/Neclow/PhD_thesis) of Neil Scheidwasser and the Phylo2Vec papers (see the references at the end).

## Imports

In [ ]:
import itertools
import math

import numpy as np

from ete4 import Tree

import phylo2vec as p2v


def plot_tree(newick):
    print(Tree(newick))

## 1. Phylogenetic trees

A phylogenetic tree is a set of nodes connected by edges (or *branches*), in which any two nodes are connected by exactly one path:
* *Leaf nodes* (or *tips*) have no descendants. They correspond to the observed taxa.
* *Internal nodes* have descendants. They represent the most recent common ancestor of the taxa that descend from them, and are not observed.
* A *cherry* is a pair of leaves that share the same parent.

A tree is *labelled* if its leaves are assigned to specific taxa, and a tree without labels or branch lengths is called a *topology*. A tree is *rooted* if its edges are directed from parent to child, and one node, the *root*, is the most recent common ancestor of all taxa. A tree is *binary* if each internal node has exactly two children.

Phylo2Vec encodes binary, rooted trees.

![A binary, rooted, labelled phylogenetic tree. Its Newick representation is (((A,B),C),D);](img/basictree.svg)

The number of possible trees grows very quickly with the number of leaves $n$: there are $(2n-3)!! = 1 \times 3 \times 5 \times \ldots \times (2n-3)$ binary, rooted, labelled trees with $n$ leaves.

In [ ]:
def n_trees(n_leaves):
    """Number of binary, rooted, labelled trees with `n_leaves` leaves: (2n - 3)!!"""
    return math.prod(range(1, 2 * n_leaves - 2, 2))


for n_leaves in [5, 10, 20, 53]:
    print(f"{n_leaves} leaves: {n_trees(n_leaves):.3g} trees")

5 leaves: 105 trees
10 leaves: 3.45e+07 trees
20 leaves: 8.2e+21 trees
53 leaves: 2.84e+82 trees


With 53 leaves, there are more possible trees than the estimated number of protons in the observable universe (about $10^{80}$). We thus need efficient data structures to store, compare and sample trees.

## 2. Why a vector?

The standard way to write trees is the Newick format, which encloses pairs of nodes (or subtrees) in nested parentheses. Newick strings are simple and flexible (they support branch lengths and other annotations), but they become hard to read and compare for large trees.

Importantly, the Newick format is not one-to-one: swapping the children of a node gives a different string for the same tree. In contrast, each tree has a unique Phylo2Vec vector:

In [ ]:
newick1 = "((0,1),(2,3));"
newick2 = "((3,2),(1,0));"  # Same tree, with swapped children

print(p2v.from_newick(newick1))
print(p2v.from_newick(newick2))

[0 2 2]
[0 2 2]


Vectors are also compact, and easy to sample and compare.

## 3. The Phylo2Vec vector

### 3.1. Node and branch labels

For a tree with $n$ leaves, Phylo2Vec uses the following conventions:
* leaves are labelled $0, \ldots, n-1$
* internal nodes are labelled $n, \ldots, 2(n-1)$, and the root is always labelled $2(n-1)$
* each branch is labelled after the node it leads to

![A tree satisfying the Phylo2Vec conventions. Its vector is (0, 0, 4, 3)](img/p2vtree.svg)

In [ ]:
v = np.array([0, 0, 4, 3])

print(p2v.to_newick(v))

plot_tree(p2v.to_newick(v))

(((0,2)6,1)7,(3,4)5)8;
   ╭─┬╴0
 ╭─┤ ╰╴2
─┤ ╰╴1
 ╰─┬╴3
   ╰╴4


### 3.2. Definition

For a tree with $n$ leaves, the vector $\boldsymbol{v}$ has $n - 1$ elements, with $v_j \in \{0, 1, \ldots, 2j\}$.

The vector describes how to build the tree by adding leaves one by one: $v_j$ is the branch from which leaf $j + 1$ descends, when it is added to the tree formed by leaves $0, \ldots, j$. This tree has $2j + 1$ branches (counting the branch above its root), hence the constraint on $v_j$. Section 5 shows this step by step.

Consequently, there are $1 \times 3 \times \ldots \times (2n - 3) = (2n-3)!!$ possible vectors: exactly as many as there are trees. Penn et al. (2024) prove that each vector corresponds to a different tree, so that the mapping between vectors and trees is one-to-one. We can check it by brute force for 5 leaves:

In [ ]:
def canonical(node):
    """String representation of a tree that does not depend on the order of children."""
    if node.is_leaf:
        return node.name
    return "(" + ",".join(sorted(canonical(child) for child in node.children)) + ")"


n_leaves = 5

all_vectors = [
    np.array(vector)
    for vector in itertools.product(*[range(2 * j + 1) for j in range(n_leaves - 1)])
]
all_trees = {canonical(Tree(p2v.to_newick(vector))) for vector in all_vectors}

print(f"{len(all_vectors)} vectors, {len(all_trees)} distinct trees")

105 vectors, 105 distinct trees


### 3.3. Ordered and unordered vectors

A vector is *ordered* if $v_j \in \{0, \ldots, j\}$ for all $j$: each new leaf descends from the branch of an existing leaf, similarly to a Yule (birth) process. There are $(n-1)!$ ordered vectors with $n$ leaves. Conversely, a vector is *completely unordered* if $v_j \in \{j + 1, \ldots, 2j\}$ for all $j \geq 1$: each new leaf descends from an internal branch.

Ordered vectors are faster to convert (see Section 8). Use `queue_shuffle` to relabel the leaves of any tree so that its vector is ordered (see [Tree operations](tree-operations.ipynb)).

In [ ]:
all_ordered = [vector for vector in all_vectors if all(vector[j] <= j for j in range(len(vector)))]

print(f"{len(all_ordered)} ordered vectors with {n_leaves} leaves; (n - 1)! = {math.factorial(n_leaves - 1)}")

24 ordered vectors with 5 leaves; (n - 1)! = 24


## 4. Other tree formats

Besides Newick strings, vectors can be converted to and from other formats that Phylo2Vec uses internally, notably in the conversion algorithms of Sections 5 and 6. We use a fixed vector so that the outputs are easier to follow:

In [ ]:
v_fixed = np.array([0, 2, 2, 5, 4, 1])

plot_tree(p2v.to_newick(v_fixed))

   ╭─┬╴0
 ╭─┤ ╰─┬╴1
 │ │   ╰╴6
─┤ ╰─┬╴4
 │   ╰╴5
 ╰─┬╴2
   ╰╴3


### 4.1. Edge list

For a tree with $n$ leaves, we can derive a list of edges of length $2 (n-1)$

Use ```to_edges``` to convert a vector to a list of tree edges (node1, node2)

In [ ]:
edges = p2v.to_edges(v_fixed)

print(edges)

[(1, 7), (6, 7), (4, 8), (5, 8), (2, 9), (3, 9), (0, 10), (7, 10), (10, 11), (8, 11), (11, 12), (9, 12)]


Use ```from_edges``` to convert a list of edges back to a vector

In [ ]:
v_fixed_from_edges = p2v.from_edges(edges)

assert np.array_equal(v_fixed, v_fixed_from_edges)  # should be the same

### 4.2. Ancestry

A visually useful format that was derived in previous versions of Phylo2Vec is the so-called "ancestry" matrix.

For a tree with $n$ leaves, the ancestry matrix is of size $(n-1, 3)$ describing a list of triplets (child1, child2, parent)

Use ```to_ancestry``` to convert a vector to an ancestry matrix

In [ ]:
ancestry = p2v.to_ancestry(v_fixed)

print(repr(ancestry))

array([[ 1,  6,  7],
       [ 4,  5,  8],
       [ 2,  3,  9],
       [ 0,  7, 10],
       [10,  8, 11],
       [11,  9, 12]])


Use ```from_ancestry``` to convert an ancestry matrix back to a vector

In [ ]:
v_fixed_from_ancestry = p2v.from_ancestry(ancestry)

assert np.array_equal(v_fixed, v_fixed_from_ancestry)  # should be the same

### 4.3. Pairs

Another intermediate format from the Phylo2Vec/Newick conversion functions is a so-called list of pairs.

For a tree with $n$ leaves, the pair list is of length $n-1$.

Each pair corresponds to a list of tuples, where each pair is represented as a tuple (B, L) indicating that leaf L descends from branch B.

Use ```to_pairs``` to convert a vector to a list of pairs

In [ ]:
pairs = p2v.to_pairs(v_fixed)

print(pairs)

[(1, 6), (4, 5), (2, 3), (0, 1), (0, 4), (0, 2)]


The pairs describe a post-order traversal of the tree. Here:
* leaves (1, 6) form a first cherry → 1 = min(1, 6) so leaf 1 is the representative of that cherry
* leaves (4, 5) form a second cherry → 4 = min(4, 5) so leaf 4 is the representative of that cherry
* leaves (2, 3) form a third cherry → 2 = min(2, 3) so leaf 2 is the representative of that cherry
* (0, 1): 1 was already visited in the first cherry, so leaf 0 forms a cherry with the parent of leaf 1
* (0, 4): both were already visited, so the parents of leaf 0 and leaf 4 form a cherry
* (0, 2): both were already visited, so the parents of leaf 0 and leaf 2 form a cherry

Use ```from_pairs``` to convert a list of pairs back to a vector

In [ ]:
v_fixed_from_pairs = p2v.from_pairs(pairs)

assert np.array_equal(v_fixed, v_fixed_from_pairs)  # should be the same

## 5. From a vector to a tree

To unpack a vector, we read it from left to right and add one leaf per element. We start from a two-leaf tree with leaves 0 and 1 and root 2, and add an extra root $R$ above it, so that the branch above the current root is also a possible placement. At step $j$, leaf $j + 1$ is added from branch $v_j$:

![Unpacking v = (0, 0, 4, 3) as a tree. Adapted from Penn et al. (2024)](img/v2newick.svg)

The trees at each step correspond to the first elements of the vector:

In [ ]:
for j in range(1, len(v) + 1):
    print(f"{repr(v[:j])}: {p2v.to_newick(v[:j])}")

array([0]): (0,1)2;
array([0, 0]): ((0,2)3,1)4;
array([0, 0, 4]): (((0,2)4,1)5,3)6;
array([0, 0, 4, 3]): (((0,2)6,1)7,(3,4)5)8;


Adding a leaf requires relabelling the internal nodes and branches. The next internal node to be labelled is the parent of the cherry with the highest label. For example, for step 3 above:

![Relabelling internal nodes after adding leaf 4 from branch 3. Adapted from Penn et al. (2024)](img/relabelling.svg)

In practice, `to_newick` works in two steps. It first builds the list of pairs (see Section 4.3), which describes a post-order traversal of the tree, then merges the pairs into a Newick string. For ordered vectors, building the pairs only requires appending to a list. For other vectors, new pairs must be inserted within the list, which is done efficiently with an [AVL tree](https://en.wikipedia.org/wiki/AVL_tree).

In [ ]:
p2v.to_pairs(v)

[(3, 4), (0, 2), (0, 1), (0, 3)]

## 6. From a Newick string to a vector

`from_newick` works in three steps.

First, the Newick string is parsed from left to right: each substring of the form `(c1,c2)p` becomes a row of the ancestry (see Section 4.2), and is replaced by its parent `p`:

![Parsing a Newick string into an ancestry](img/parse.svg)

Second, the ancestry is sorted to match the order of the pairs. If the Newick string has parent labels, each row is placed according to its parent label. Otherwise, the parent of each cherry is replaced by the maximum of its two children during parsing, and the minimum is kept to represent the cherry. The rows are then sorted using the fact that two cherries with the same minimum appear in post-order when the Newick string is read from left to right:

![Sorting an ancestry derived from a Newick string without parent labels](img/order_cherries_no_parents.svg)

Third, the vector is built from the sorted ancestry. In each cherry $(c_m, c_M)$, $c_M$ is the leaf that was added, and $c_m$ its smallest descendant. A [Fenwick tree](https://en.wikipedia.org/wiki/Fenwick_tree) $\boldsymbol{f}$ counts the cherries processed so far whose added leaf is smaller than $c_M$. If there are none, $c_M$ descends from the branch of leaf $c_m$, and $v_{c_M - 1} = c_m$. Otherwise, $c_M$ descends from an internal branch, and $v_{c_M - 1} = c_M - 1 + f_\Sigma(c_M - 1)$, where $f_\Sigma$ is the prefix sum of $\boldsymbol{f}$:

![Building a vector from a sorted ancestry](img/build_vector.svg)

Both Newick strings, with and without parent labels, give the same vector:

In [ ]:
from phylo2vec.utils.newick import remove_parent_labels

newick = p2v.to_newick(v)
newick_no_parents = remove_parent_labels(newick)

print(f"{newick} -> {p2v.from_newick(newick)}")
print(f"{newick_no_parents} -> {p2v.from_newick(newick_no_parents)}")

(((0,2)6,1)7,(3,4)5)8; -> [0 0 4 3]
(((0,2),1),(3,4)); -> [0 0 4 3]


## 7. Branch lengths: the matrix format

To store branch lengths, the vector is extended into a matrix with two extra columns. Branch lengths are attributed to edges using the ancestry: row $j$ of the matrix holds $v_j$ and the lengths of the branches leading to the two children in row $j$ of the ancestry. In the figure below, the matrix is shown transposed: in the code, it has shape $(n - 1, 3)$.

![Annotating v = (0, 0, 4, 3) with branch lengths. Adapted from Scheidwasser et al. (2025)](img/phylo2mat.svg)

To see where each branch length goes, we can set the length of the branch leading to node $i$ to $i / 10 + 0.01$:

In [ ]:
newick_with_bls = "(((0:0.01,2:0.21)6:0.61,1:0.11)7:0.71,(3:0.31,4:0.41)5:0.51)8;"

m = p2v.from_newick(newick_with_bls)

print(f"ancestry:\n{p2v.to_ancestry(v)}")
print(f"matrix:\n{m}")

ancestry:
[[3 4 5]
 [0 2 6]
 [6 1 7]
 [7 5 8]]
matrix:
[[0.   0.31 0.41]
 [0.   0.01 0.21]
 [4.   0.61 0.11]
 [3.   0.71 0.51]]


As for vectors, the matrix does not depend on the order of children in the Newick string:

In [ ]:
m1 = p2v.from_newick("((0:0.1,1:0.2)3:0.3,2:0.4)4;")
m2 = p2v.from_newick("(2:0.4,(1:0.2,0:0.1)3:0.3)4;")

np.array_equal(m1, m2)

True

## 8. Complexity

For a tree with $n$ leaves:

| Conversion | Step | Time complexity |
|---|---|---|
| Vector → Newick | Build the pairs | $\mathcal{O}(n)$ for ordered vectors, $\mathcal{O}(n \log n)$ otherwise |
| | Build the Newick string from the pairs | $\mathcal{O}(n \sqrt{n})$ on average, from $\mathcal{O}(n \log n)$ (balanced trees) to $\mathcal{O}(n^2)$ (ladder trees) |
| Newick → vector | Parse the Newick string | $\mathcal{O}(n)$ |
| | Sort the ancestry | $\mathcal{O}(n)$ with parent labels, $\mathcal{O}(n \log n)$ without |
| | Build the vector | $\mathcal{O}(n \log n)$ |

## References

* Penn, M. J., Scheidwasser, N., Khurana, M. P., Duchêne, D. A., Donnelly, C. A., & Bhatt, S. (2024). Phylo2Vec: a vector representation for binary trees. *Systematic Biology*. <https://doi.org/10.1093/sysbio/syae030>
* Scheidwasser, N., Nag, A., Penn, M. J., Jakob, A., Andersen, F. M., Khurana, M. P., Setiawan, L., Duchêne, D. A., & Bhatt, S. (2025). phylo2vec: a library for vector-based phylogenetic tree manipulation. *Journal of Open Source Software*. <https://doi.org/10.21105/joss.09040>
* Scheidwasser, N. PhD thesis. <https://github.com/Neclow/PhD_thesis>